<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/triton_GLM_topo_cbp_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q

In [1]:
!nvidia-smi

Sun Nov 22 14:59:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/

total 372G
drwx------ 51 root root 4.0K Nov 22 15:00 ..
-rw-------  1 root root 2.3K Nov 22 14:29 GLM-4-32B-0414-topo-triton-Q8_0_audit.json
-rw-------  1 root root  33G Nov 22 14:19 GLM-4-32B-0414-topo-triton-Q8_0.gguf
-rw-------  1 root root 8.9K Nov 21 14:30 somala_drive_gguf_audit.json
-rw-------  1 root root  26G Nov 21 05:08 Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf
-rw-------  1 root root 7.2G Nov 19 03:18 mistral-7b-text-to-sql-topo-Q8_0.gguf
-rw-------  1 root root  27G Nov 14 20:14 qwen3.8-27b-topocbp-Q8_0.gguf
-rw-------  1 root root  11G Nov 12 06:55 deepseek_moe_16b_topocbp_q4_k_m.gguf
-rw-------  1 root root  70G Nov  9 06:01 deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf
-rw-------  1 root root  32G Nov  9 05:57 sarvam-30b-topo-governed-q8_0.gguf
-rw-------  1 root root  16G Nov  9 04:40 muse-glimmer-30b-card-faithful-Q4_K_M.gguf
-rw-------  1 root root  15G Nov  9 04:36 gpt-oss-20b-topo-governed-Q4_K_M.gguf
-rw-------  1 root root  15G Nov  9 04:34 certified-

In [5]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.6/893.6 MB 50.3 MB/s  0:00:17
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


## train

In [6]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import math
import hashlib
import random
import torch
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset

try:
    import triton
    import triton.language as tl

    HAS_TRITON = True
except ImportError:
    triton = None
    tl = None
    HAS_TRITON = False

# =====================================================================
# 0. SEED & DEVICE
# =====================================================================
SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
use_cuda = num_gpus > 0
primary_device = torch.device("cuda:0" if use_cuda else "cpu")

# =====================================================================
# 1. TOPO-2026 OPTIMIZER WITH FUSED TRITON KERNEL & PYTORCH FALLBACK
# =====================================================================
SAFETY_CONSTANT = 0.85
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]


def _stochastic_round_(src_fp32: torch.Tensor, dst_bf16: torch.Tensor) -> None:
    bits = src_fp32.view(torch.int32)
    bits.add_(torch.randint_like(bits, 0, 1 << 16))
    bits.bitwise_and_(-65536)
    dst_bf16.copy_(src_fp32)


if HAS_TRITON:

    @triton.jit
    def _topo_adamw_fused_step_kernel(
        p_ptr,          # *mut bf16 or fp32
        grad_ptr,       # *const bf16 or fp32
        m_buf_ptr,      # *mut bfloat16
        v_buf_ptr,      # *mut bfloat16
        comp_ptr,       # *mut bfloat16
        n_elements,     # int
        lr,             # float32
        beta1,          # float32
        beta2,          # float32
        eps,            # float32
        wd,             # float32
        bias_corr1,     # float32 (1.0 - beta1**t)
        bias_corr2,     # float32 (1.0 - beta2**t)
        seed_m,         # int32 seed for m stochastic rounding
        seed_v,         # int32 seed for v stochastic rounding
        P_IS_BF16: tl.constexpr,
        BLOCK_SIZE: tl.constexpr,
    ):
        pid = tl.program_id(axis=0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements

        p_raw = tl.load(p_ptr + offsets, mask=mask, other=0.0)
        g = tl.load(grad_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        m_prev = tl.load(m_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        v_prev = tl.load(v_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        comp_prev = tl.load(comp_ptr + offsets, mask=mask, other=0.0).to(tl.float32)

        w = p_raw.to(tl.float32)

        # Moment updates
        m = beta1 * m_prev + (1.0 - beta1) * g
        v = beta2 * v_prev + (1.0 - beta2) * g * g

        # Bias-corrected AdamW step + decoupled weight decay + error compensation
        m_hat = m / bias_corr1
        v_hat = v / bias_corr2
        denom = tl.sqrt(v_hat) + eps
        step_update = -lr * (m_hat / denom + wd * w) + comp_prev

        total = w + step_update
        if P_IS_BF16:
            new_p = total.to(tl.bfloat16)
            new_comp = total - new_p.to(tl.float32)
            tl.store(p_ptr + offsets, new_p, mask=mask)
            tl.store(comp_ptr + offsets, new_comp.to(tl.bfloat16), mask=mask)
        else:
            tl.store(p_ptr + offsets, total, mask=mask)
            tl.store(comp_ptr + offsets, 0.0, mask=mask)

        # Cast to tl.int32 before &-65536 because tl.randint returns tl.uint32
        m_bits = m.to(tl.int32, bitcast=True)
        m_jitter = tl.randint(seed_m, offsets) & 0xFFFF
        m_rounded_bits = (m_bits + m_jitter).to(tl.int32) & -65536
        m_rounded = m_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        v_bits = v.to(tl.int32, bitcast=True)
        v_jitter = tl.randint(seed_v, offsets) & 0xFFFF
        v_rounded_bits = (v_bits + v_jitter).to(tl.int32) & -65536
        v_rounded = v_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        tl.store(m_buf_ptr + offsets, m_rounded, mask=mask)
        tl.store(v_buf_ptr + offsets, v_rounded, mask=mask)


def _triton_param_step(
    p: torch.Tensor,
    grad: torch.Tensor,
    m_buf: torch.Tensor,
    v_buf: torch.Tensor,
    comp: torch.Tensor,
    lr: float,
    beta1: float,
    beta2: float,
    eps: float,
    wd: float,
    step_t: int,
    base_seed: int = SEED,
    block_size: int = 1024,
) -> None:
    n_elements = p.numel()
    if n_elements == 0:
        return

    p_c = p.contiguous().view(-1)
    g_c = grad.contiguous().view(-1)
    m_c = m_buf.contiguous().view(-1)
    v_c = v_buf.contiguous().view(-1)
    c_c = comp.contiguous().view(-1)

    bias_corr1 = 1.0 - (beta1 ** step_t)
    bias_corr2 = 1.0 - (beta2 ** step_t)
    seed_m = int((base_seed * 1_000_003 + step_t * 97) & 0x7FFFFFFF)
    seed_v = int((base_seed * 1_000_033 + step_t * 193) & 0x7FFFFFFF)

    grid = (triton.cdiv(n_elements, block_size),)
    _topo_adamw_fused_step_kernel[grid](
        p_c,
        g_c,
        m_c,
        v_c,
        c_c,
        n_elements,
        float(lr),
        float(beta1),
        float(beta2),
        float(eps),
        float(wd),
        float(bias_corr1),
        float(bias_corr2),
        seed_m,
        seed_v,
        P_IS_BF16=(p_c.dtype == torch.bfloat16),
        BLOCK_SIZE=block_size,
    )

    if not p.is_contiguous():
        p.copy_(p_c.view_as(p))
    if not m_buf.is_contiguous():
        m_buf.copy_(m_c.view_as(m_buf))
    if not v_buf.is_contiguous():
        v_buf.copy_(v_c.view_as(v_buf))
    if not comp.is_contiguous():
        comp.copy_(c_c.view_as(comp))


def _pytorch_param_step(
    p: torch.Tensor,
    m_buf: torch.Tensor,
    v_buf: torch.Tensor,
    comp: torch.Tensor,
    lr: float,
    beta1: float,
    beta2: float,
    eps: float,
    wd: float,
    step_t: int,
) -> None:
    g = p.grad.float()
    m = m_buf.float().mul_(beta1).add_(g, alpha=1.0 - beta1)
    v = v_buf.float().mul_(beta2).addcmul_(g, g, value=1.0 - beta2)
    del g

    w = p.float()
    denom = (v.sqrt() / math.sqrt(1.0 - beta2 ** step_t)).add_(eps)
    y = m.div(denom).mul_(-lr / (1.0 - beta1 ** step_t))
    del denom

    y.add_(w, alpha=-lr * wd).add_(comp.float())
    _stochastic_round_(m, m_buf)
    _stochastic_round_(v, v_buf)
    del m, v

    total = w.add_(y)
    new = total.to(torch.bfloat16)
    comp.copy_(total.sub_(new.float()))
    if p.dtype == torch.bfloat16:
        p.copy_(new)
    else:
        p.copy_(new.to(p.dtype))
    del y, w, total, new


class TopoAdamW(Optimizer):
    def __init__(
        self,
        params,
        lr=1e-4,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-2,
        embedding_layer=None,
        prime_anchors=None,
        lambda_euler=SAFETY_CONSTANT,
        base_seed=SEED,
        use_triton=True,
    ):
        if embedding_layer is None:
            raise ValueError("'embedding_layer' must be passed explicitly.")
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

        self.embedding_layer = embedding_layer
        self.lambda_euler = lambda_euler
        self.base_seed = base_seed
        self.use_triton = bool(use_triton and HAS_TRITON)

        w = embedding_layer.weight.data
        self.valid_anchors = [i for i in (prime_anchors or PRIME_ANCHORS) if i < w.shape[0]]
        self.baseline = {i: w[i].detach().clone() for i in self.valid_anchors}
        self.certified_sha256_digest = self._digest()
        print(
            f"[TopoAdamW] Baseline locked (Triton={self.use_triton}). "
            f"Digest: {self.certified_sha256_digest[:16]}"
        )

    def _digest(self) -> str:
        w = self.embedding_layer.weight.data
        if not self.valid_anchors:
            return hashlib.sha256(b"").hexdigest()
        buf = torch.cat([w[i].flatten().float() for i in self.valid_anchors]).cpu().numpy().tobytes()
        return hashlib.sha256(buf).hexdigest()

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        emb_w = self.embedding_layer.weight
        if self.valid_anchors and self.baseline[self.valid_anchors[0]].device != emb_w.device:
            self.baseline = {i: t.to(emb_w.device) for i, t in self.baseline.items()}
        if emb_w.grad is not None:
            for i in self.valid_anchors:
                emb_w.grad[i].zero_()

        for group in self.param_groups:
            lr, wd, eps = group["lr"], group["weight_decay"], group["eps"]
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if not p.requires_grad or p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["exp_avg_sq"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["comp"] = torch.zeros_like(p, dtype=torch.bfloat16)

                state["step"] += 1
                t = state["step"]
                m_buf, v_buf, comp = state["exp_avg"], state["exp_avg_sq"], state["comp"]
                grad_on_dev = p.grad.to(p.device) if p.grad.device != p.device else p.grad

                if self.use_triton and p.is_cuda and grad_on_dev.is_cuda:
                    _triton_param_step(
                        p=p,
                        grad=grad_on_dev,
                        m_buf=m_buf,
                        v_buf=v_buf,
                        comp=comp,
                        lr=lr,
                        beta1=beta1,
                        beta2=beta2,
                        eps=eps,
                        wd=wd,
                        step_t=t,
                        base_seed=self.base_seed,
                    )
                else:
                    orig_grad = p.grad
                    p.grad = grad_on_dev
                    try:
                        _pytorch_param_step(
                            p=p,
                            m_buf=m_buf,
                            v_buf=v_buf,
                            comp=comp,
                            lr=lr,
                            beta1=beta1,
                            beta2=beta2,
                            eps=eps,
                            wd=wd,
                            step_t=t,
                        )
                    finally:
                        p.grad = orig_grad

                if p is emb_w:
                    for i in self.valid_anchors:
                        p[i].copy_(self.baseline[i].to(device=p.device, dtype=p.dtype))
                        comp[i].zero_()

        return loss

    def audit_telemetry(self):
        w = self.embedding_layer.weight.data
        drift = (
            max(
                torch.max(torch.abs(w[i].float() - self.baseline[i].to(w.device).float())).item()
                for i in self.valid_anchors
            )
            if self.valid_anchors
            else 0.0
        )
        h = self._digest()
        return drift, h, h == self.certified_sha256_digest


# =====================================================================
# 2. MODEL / TOKENIZER (zai-org/GLM-4-32B-0414)
# =====================================================================
model_id = "zai-org/GLM-4-32B-0414"
max_seq_len = 512

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
    device_map={"": 0} if use_cuda else "auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.config.use_cache = False

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

for p in model.parameters():
    p.requires_grad = False
for name, mod in model.named_modules():
    if "lm_head" in name or "norm" in name:
        for p in mod.parameters(recurse=False):
            p.requires_grad = True

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"[*] Trainable parameters: {sum(p.numel() for p in trainable_params):,}")

topo_optimizer = TopoAdamW(
    trainable_params,
    lr=1e-4,
    weight_decay=1e-2,
    embedding_layer=model.get_input_embeddings(),
    prime_anchors=PRIME_ANCHORS,
    base_seed=SEED,
    use_triton=True,
)


def _get_input_device(m) -> torch.device:
    emb = m.get_input_embeddings()
    if emb is not None and hasattr(emb, "weight") and emb.weight.device.type != "meta":
        return emb.weight.device
    return primary_device


input_device = _get_input_device(model)
print(f"[*] Input target device: {input_device}")

# =====================================================================
# 3. DATA
# =====================================================================
def build_user_prompt(sample):
    instruction = (sample.get("instruction") or "").strip()
    extra_input = (sample.get("input") or "").strip()
    return f"{instruction}\n\nInput:\n{extra_input}" if extra_input else instruction


def preprocess_function(sample):
    user = build_user_prompt(sample)
    if getattr(tokenizer, "chat_template", None):
        try:
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "user", "content": user}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "user", "content": user}],
                tokenize=False,
                add_generation_prompt=True,
            )
    else:
        bos = tokenizer.bos_token or ""
        prompt_text = f"{bos}{user}\n"

    eos_token = tokenizer.eos_token or ""
    answer_text = (sample.get("output") or "") + eos_token
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]
    input_ids = (list(prompt_ids) + answer_ids)[:max_seq_len]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:max_seq_len]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


raw = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=SEED)
train_ds = raw.select(range(12500)).map(preprocess_function, remove_columns=raw.column_names)
eval_ds = raw.select(range(12500, 13000)).map(preprocess_function, remove_columns=raw.column_names)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
)
train_loader = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
    pin_memory=(use_cuda and input_device.type == "cuda"),
)
eval_loader = DataLoader(
    eval_ds,
    batch_size=2,
    shuffle=False,
    collate_fn=collator,
    pin_memory=(use_cuda and input_device.type == "cuda"),
)

# =====================================================================
# 4. PURE PYTORCH + TRITON TRAINING & AUDIT LOOP
# =====================================================================
epochs = 3
grad_accum = 16
max_grad_norm = 1.0
total_steps = (len(train_loader) // grad_accum) * epochs
scheduler = get_cosine_schedule_with_warmup(
    topo_optimizer, num_warmup_steps=20, num_training_steps=max(total_steps, 1)
)


@torch.no_grad()
def evaluate_loss():
    model.eval()
    losses = []
    for batch in eval_loader:
        batch = {
            k: v.to(input_device, non_blocking=(use_cuda and input_device.type == "cuda"))
            for k, v in batch.items()
        }
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_cuda):
            out = model(**batch)
        losses.append(out.loss.item())
    model.train()
    return sum(losses) / max(len(losses), 1)


print(f"[baseline] validation loss before training: {evaluate_loss():.4f}", flush=True)

output_dir = "/content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train"
os.makedirs(output_dir, exist_ok=True)

model.train()
topo_optimizer.zero_grad()
global_step = 0
running_step_loss = 0.0
accum_loss = 0.0

for epoch in range(1, epochs + 1):
    for step_idx, batch in enumerate(train_loader, start=1):
        batch = {
            k: v.to(input_device, non_blocking=(use_cuda and input_device.type == "cuda"))
            for k, v in batch.items()
        }
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_cuda):
            outputs = model(**batch)
            loss = outputs.loss / grad_accum

        loss.backward()
        accum_loss += outputs.loss.detach().item() / grad_accum

        if step_idx % grad_accum == 0:
            active_grads = [p for p in trainable_params if p.grad is not None]
            if active_grads:
                torch.nn.utils.clip_grad_norm_(active_grads, max_grad_norm)
            topo_optimizer.step()
            scheduler.step()
            topo_optimizer.zero_grad()

            global_step += 1
            running_step_loss += accum_loss
            accum_loss = 0.0

            if global_step % 20 == 0:
                avg_loss = running_step_loss / 20.0
                lr_now = scheduler.get_last_lr()[0]
                print(f"step {global_step:>5} | train loss {avg_loss:.4f} | lr {lr_now:.2e}", flush=True)
                running_step_loss = 0.0

            if global_step % 100 == 0:
                val_loss = evaluate_loss()
                print(f"[eval] step {global_step:>5} | val loss {val_loss:.4f}", flush=True)

    epoch_dir = os.path.join(output_dir, f"epoch-{epoch}")
    os.makedirs(epoch_dir, exist_ok=True)
    model.save_pretrained(epoch_dir)
    tokenizer.save_pretrained(epoch_dir)
    print(f"[*] Saved epoch {epoch} checkpoint -> {epoch_dir}", flush=True)

max_drift, current_hash, hash_matched = topo_optimizer.audit_telemetry()
print(f"[TopoAudit] Max Anchor Drift: {max_drift:.6e} | Hash Matched: {hash_matched} | Digest: {current_hash[:16]}")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

... (17 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[*] Saved epoch 3 checkpoint -> /content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train/epoch-3
[TopoAudit] Max Anchor Drift: 0.000000e+00 | Hash Matched: True | Digest: d6b0458680af9d90


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train/tokenizer_config.json',
 '/content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train/chat_template.jinja',
 '/content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train/tokenizer.json')

In [9]:
!ls -ltha /content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train/

total 19G
drwx------  5 root root 4.0K Nov 22 13:28 .
-rw-------  1 root root 1015 Nov 22 13:28 chat_template.jinja
-rw-------  1 root root  699 Nov 22 13:28 tokenizer_config.json
-rw-------  1 root root  20M Nov 22 13:28 tokenizer.json
-rw-------  1 root root  19G Nov 22 13:28 model.safetensors
-rw-------  1 root root 1.3K Nov 22 13:27 config.json
drwx------  2 root root 4.0K Nov 22 13:27 epoch-3
-rw-------  1 root root  155 Nov 22 13:27 generation_config.json
drwx------  2 root root 4.0K Nov 22 12:25 epoch-2
drwx------  2 root root 4.0K Nov 22 11:23 epoch-1
drwx------ 26 root root 4.0K Nov 22 10:22 ..


In [10]:
%cd /content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train/
!du -sh *

/content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train
1.0K	chat_template.jinja
1.5K	config.json
19G	epoch-1
19G	epoch-2
19G	epoch-3
512	generation_config.json
19G	model.safetensors
1.0K	tokenizer_config.json
20M	tokenizer.json


In [11]:
!nvidia-smi

Sun Nov 22 13:34:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             85W /  600W |   31671MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [12]:
import gc
import hashlib
import os
import torch
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_ID = "zai-org/GLM-4-32B-0414"
CKPT_DIR = "/content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train"
MERGED_BF16_DIR = "/content/drive/MyDrive/topogpt/GLM-4-32B-0414-topo-triton-bf16-merged"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
EXPECTED_DIGEST = "d6b0458680af9d90"

os.makedirs(MERGED_BF16_DIR, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR, trust_remote_code=True)

# Load base in bfloat16 on CPU (or multi-GPU if available) so 32B weights don't OOM a single GPU
model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    dtype=torch.bfloat16,
    device_map="cpu",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Overlay trained norm / lm_head tensors from saved checkpoint if present
trained_tensors = {}
for fname in os.listdir(CKPT_DIR):
    fpath = os.path.join(CKPT_DIR, fname)
    if fname.endswith(".safetensors"):
        trained_tensors.update(load_file(fpath, device="cpu"))
    elif fname.endswith(".bin") or fname.endswith(".pt"):
        sd = torch.load(fpath, map_location="cpu")
        if isinstance(sd, dict):
            trained_tensors.update(sd)

state_dict = model.state_dict()
overlaid = 0
for k, v in trained_tensors.items():
    if k in state_dict and state_dict[k].shape == v.shape:
        state_dict[k].copy_(v.to(dtype=state_dict[k].dtype))
        overlaid += 1

print(f"[*] Overlaid {overlaid} matching parameter tensors from {CKPT_DIR}")

# Audit prime-anchor SHA-256 digest on input embeddings
emb_w = model.get_input_embeddings().weight.data
valid_anchors = [i for i in PRIME_ANCHORS if i < emb_w.shape[0]]
raw_bytes = torch.cat([emb_w[i].flatten().float() for i in valid_anchors]).cpu().numpy().tobytes()
digest = hashlib.sha256(raw_bytes).hexdigest()
print(f"[Audit] Digest: {digest[:16]} | Prefix Match: {digest.startswith(EXPECTED_DIGEST)}")

model.save_pretrained(MERGED_BF16_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_BF16_DIR)
print(f"[*] Saved merged BF16 model -> {MERGED_BF16_DIR}")

del model, trained_tensors
gc.collect()

Loading weights:   0%|          | 0/613 [00:00<?, ?it/s]

[*] Overlaid 247 matching parameter tensors from /content/drive/MyDrive/topogpt/GLM-4-32B-0414-alpaca-topo-4bit-triton-train
[Audit] Digest: d6b0458680af9d90 | Prefix Match: True


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

[*] Saved merged BF16 model -> /content/drive/MyDrive/topogpt/GLM-4-32B-0414-topo-triton-bf16-merged


5351

In [ ]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


## gguf - convert

In [13]:
import os
import subprocess
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
MERGED_BF16_DIR = "/content/drive/MyDrive/topogpt/GLM-4-32B-0414-topo-triton-bf16-merged"
GGUF_DIR = "/content/drive/MyDrive/GGUF_Models"
F16_OUT = os.path.join(GGUF_DIR, "GLM-4-32B-0414-topo-triton-F16.gguf")

os.makedirs(GGUF_DIR, exist_ok=True)

subprocess.run(
    [
        "python3",
        os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"),
        MERGED_BF16_DIR,
        "--outfile",
        F16_OUT,
        "--outtype",
        "f16",
    ],
    check=True,
)

print(f"[OK] F16 GGUF created: {F16_OUT} ({os.path.getsize(F16_OUT) / (1024**3):.2f} GiB)")

[OK] F16 GGUF created: /content/drive/MyDrive/GGUF_Models/GLM-4-32B-0414-topo-triton-F16.gguf (60.67 GiB)


## gguf - quantize

In [14]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


In [15]:
import os
import subprocess
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
GGUF_DIR = "/content/drive/MyDrive/GGUF_Models"
F16_IN = os.path.join(GGUF_DIR, "GLM-4-32B-0414-topo-triton-F16.gguf")
Q8_OUT = os.path.join(GGUF_DIR, "GLM-4-32B-0414-topo-triton-Q8_0.gguf")

quantize_candidates = [
    os.path.join(LLAMA_CPP_DIR, "build", "bin", "llama-quantize"),
    os.path.join(LLAMA_CPP_DIR, "llama-quantize"),
]
QUANTIZE_BIN = next((p for p in quantize_candidates if os.path.isfile(p)), quantize_candidates[0])

lib_dirs = [
    os.path.join(LLAMA_CPP_DIR, "build", "bin"),
    os.path.join(LLAMA_CPP_DIR, "build", "lib"),
    os.path.join(LLAMA_CPP_DIR, "build"),
]
existing_lib_dirs = [d for d in lib_dirs if os.path.isdir(d)]

env = os.environ.copy()
prev_ld = env.get("LD_LIBRARY_PATH", "")
env["LD_LIBRARY_PATH"] = ":".join(existing_lib_dirs + ([prev_ld] if prev_ld else []))

subprocess.run([QUANTIZE_BIN, F16_IN, Q8_OUT, "Q8_0"], env=env, check=True)

print(f"[OK] Q8_0 GGUF created: {Q8_OUT} ({os.path.getsize(Q8_OUT) / (1024**3):.2f} GiB)")

[OK] Q8_0 GGUF created: /content/drive/MyDrive/GGUF_Models/GLM-4-32B-0414-topo-triton-Q8_0.gguf (32.24 GiB)


## gguf - eval

In [1]:
import hashlib
import json
import os
import torch
from llama_cpp import Llama
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

GGUF_PATH = "/content/drive/MyDrive/GGUF_Models/GLM-4-32B-0414-topo-triton-Q8_0.gguf"
MERGED_BF16_DIR = "/content/drive/MyDrive/topogpt/GLM-4-32B-0414-topo-triton-bf16-merged"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
EXPECTED_DIGEST = "d6b0458680af9d90"
SEED = 123

# 1. Verify merged BF16 / reference prime-anchor SHA-256 baseline if available
ref_digest = None
ref_match = None
if os.path.isdir(MERGED_BF16_DIR):
    from safetensors.torch import load_file

    emb_tensor = None
    for fname in sorted(os.listdir(MERGED_BF16_DIR)):
        if fname.endswith(".safetensors"):
            sd = load_file(os.path.join(MERGED_BF16_DIR, fname), device="cpu")
            for k, v in sd.items():
                if "embed_tokens" in k or "word_embeddings" in k:
                    emb_tensor = v
                    break
        if emb_tensor is not None:
            break

    if emb_tensor is not None:
        valid_anchors = [i for i in PRIME_ANCHORS if i < emb_tensor.shape[0]]
        raw_bytes = (
            torch.cat([emb_tensor[i].flatten().float() for i in valid_anchors])
            .cpu()
            .numpy()
            .tobytes()
        )
        ref_digest = hashlib.sha256(raw_bytes).hexdigest()
        ref_match = ref_digest.startswith(EXPECTED_DIGEST)
        print(
            f"[Reference Anchor Audit] Digest: {ref_digest[:16]} | "
            f"Matches Baseline ({EXPECTED_DIGEST}): {ref_match}"
        )

# 2. Load Q8_0 GGUF via llama-cpp-python
use_cuda = torch.cuda.is_available()
n_gpu_layers = -1 if use_cuda else 0
n_threads = None if use_cuda else 1

llm = Llama(
    model_path=GGUF_PATH,
    n_ctx=2048,
    n_gpu_layers=n_gpu_layers,
    n_threads=n_threads,
    seed=SEED,
    verbose=False,
)


def run_deterministic(prompt_text: str, max_tokens: int = 320) -> str:
    llm.reset()
    out = llm(
        prompt_text,
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=1.0,
        top_k=1,
        repeat_penalty=1.0,
        seed=SEED,
    )
    return out["choices"][0]["text"].strip()


# 3. Run deterministic prompts twice to confirm zero run-to-run variance
prompts = [
    "Explain the stability-plasticity dilemma in continual learning in three concise bullet points.",
    "Write a short SQL query to find duplicate customer emails in a users table.",
]

results = []
for prompt in prompts:
    text_a = run_deterministic(prompt, max_tokens=320)
    text_b = run_deterministic(prompt, max_tokens=320)
    exact_repeat = text_a == text_b
    out_hash = hashlib.sha256(text_a.encode("utf-8")).hexdigest()[:16]

    print(f"\n--- Prompt: {prompt}")
    print(f"Deterministic Repeat Match: {exact_repeat} | Output SHA-256[:16]: {out_hash}")
    print(text_a)

    results.append(
        {
            "prompt": prompt,
            "deterministic_repeat": exact_repeat,
            "output_sha256_16": out_hash,
            "response": text_a,
        }
    )

report_path = "/content/drive/MyDrive/GGUF_Models/GLM-4-32B-0414-topo-triton-Q8_0_audit.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "gguf_path": GGUF_PATH,
            "baseline_digest": EXPECTED_DIGEST,
            "reference_digest": ref_digest,
            "reference_match": ref_match,
            "seed": SEED,
            "n_gpu_layers": n_gpu_layers,
            "n_threads": n_threads,
            "results": results,
        },
        f,
        indent=2,
    )
print(f"\n[OK] Saved verification report -> {report_path}")

[Reference Anchor Audit] Digest: d6b0458680af9d90 | Matches Baseline (d6b0458680af9d90): True

--- Prompt: Explain the stability-plasticity dilemma in continual learning in three concise bullet points.
Deterministic Repeat Match: True | Output SHA-256[:16]: 0568fcad58d98807
- The stability-plasticity dilemma refers to the challenge of balancing the need to retain previously learned information (stability) while also being able to learn new information (plasticity) in a continually changing environment. - In a stable environment, it is important to maintain consistency and stability in order to retain and build upon existing knowledge. However, in a dynamic environment, it is necessary to be plastic and adaptable in order to learn and incorporate new information. - This dilemma arises because the mechanisms that enable plasticity, such as neural network weight changes, can also cause instability and interfere with the retention of previously learned information. Therefore, finding the r

## hf

In [2]:
import os
from google.colab import drive, userdata
from huggingface_hub import HfApi, login

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

GGUF_PATH = "/content/drive/MyDrive/GGUF_Models/GLM-4-32B-0414-topo-triton-Q8_0.gguf"
AUDIT_PATH = "/content/drive/MyDrive/GGUF_Models/GLM-4-32B-0414-topo-triton-Q8_0_audit.json"
REPO_ID = "frankmorales2020/GLM-4-32B-0414-topo-triton-Q8_0-GGUF"

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True, private=False)

print(f"[*] Uploading {GGUF_PATH} -> {REPO_ID} ...")
api.upload_file(
    path_or_fileobj=GGUF_PATH,
    path_in_repo=os.path.basename(GGUF_PATH),
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload GLM-4-32B-0414 TopoAdamW Triton Q8_0 GGUF (digest d6b0458680af9d90)",
)

if os.path.isfile(AUDIT_PATH):
    api.upload_file(
        path_or_fileobj=AUDIT_PATH,
        path_in_repo=os.path.basename(AUDIT_PATH),
        repo_id=REPO_ID,
        repo_type="model",
        commit_message="Add Q8_0 deterministic & prime-anchor audit report",
    )

print(f"[OK] Upload complete: https://huggingface.co/{REPO_ID}")

[*] Uploading /content/drive/MyDrive/GGUF_Models/GLM-4-32B-0414-topo-triton-Q8_0.gguf -> frankmorales2020/GLM-4-32B-0414-topo-triton-Q8_0-GGUF ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...414-topo-triton-Q8_0.gguf:   0%|          | 1.09MB / 34.6GB            

Validating                    :           |   0%            

[OK] Upload complete: https://huggingface.co/frankmorales2020/GLM-4-32B-0414-topo-triton-Q8_0-GGUF


## inference-hf

In [3]:
import hashlib
import os
import torch
from google.colab import userdata
from huggingface_hub import hf_hub_download, login
from llama_cpp import Llama

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

REPO_ID = "frankmorales2020/GLM-4-32B-0414-topo-triton-Q8_0-GGUF"
FILENAME = "GLM-4-32B-0414-topo-triton-Q8_0.gguf"
SEED = 123

# 1. Download (or reuse cached) GGUF directly from Hugging Face Hub
gguf_local_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"[*] Using GGUF from HF cache: {gguf_local_path}")

# 2. Load with llama-cpp-python
use_cuda = torch.cuda.is_available()
n_gpu_layers = -1 if use_cuda else 0
n_threads = None if use_cuda else 1

llm = Llama(
    model_path=gguf_local_path,
    n_ctx=2048,
    n_gpu_layers=n_gpu_layers,
    n_threads=n_threads,
    seed=SEED,
    verbose=False,
)


def run_test(prompt_text: str, max_tokens: int = 256) -> str:
    llm.reset()
    out = llm(
        prompt_text,
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=1.0,
        top_k=1,
        repeat_penalty=1.0,
        seed=SEED,
    )
    return out["choices"][0]["text"].strip()


# 3. Run inference test prompts and verify determinism
test_prompts = [
    "Summarize the main trade-off in continual learning between retaining past knowledge and adapting to new data.",
    "Write a concise Python function that computes the SHA-256 hex digest of a string.",
]

for prompt in test_prompts:
    resp_1 = run_test(prompt, max_tokens=256)
    resp_2 = run_test(prompt, max_tokens=256)
    matched = resp_1 == resp_2
    digest_16 = hashlib.sha256(resp_1.encode("utf-8")).hexdigest()[:16]

    print(f"\n=== Prompt: {prompt}")
    print(f"Repeat Match: {matched} | SHA-256[:16]: {digest_16}")
    print(resp_1)

GLM-4-32B-0414-topo-triton-Q8_0.gguf: reconstructing file:   0%|          |  0.00B / 34.6GB            

GLM-4-32B-0414-topo-triton-Q8_0.gguf: downloading bytes:           |  0.00B            

[*] Using GGUF from HF cache: /root/.cache/huggingface/hub/models--frankmorales2020--GLM-4-32B-0414-topo-triton-Q8_0-GGUF/snapshots/14e13bdefef0b0499f476fdccd84e8a9bb1bf02a/GLM-4-32B-0414-topo-triton-Q8_0.gguf

=== Prompt: Summarize the main trade-off in continual learning between retaining past knowledge and adapting to new data.
Repeat Match: True | SHA-256[:16]: 0c7d20f396ac6a38
How do different approaches attempt to balance this trade-off?
The main trade-off in continual learning is the balance between retaining past knowledge and adapting to new data. This is often referred to as the "catastrophic forgetting" problem, where the model's performance on previously learned tasks degrades significantly when it is trained on new tasks. Different approaches attempt to balance this trade-off in various ways:

1. Regularization-based methods: These methods add a penalty term to the loss function to encourage the model to retain past knowledge. For example, elastic weight consolidation (EWC